In [ ]:
# code ask the user to input city name first to check which city we are looking for: 
import requests
import math
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import numpy as np

# GoMaps API Key
API_KEY = "AlzaSyOkVMhcZDvHkm2ykwxh51Z-EM_wbk1mAWx"  # Replace with your actual API key

# Function to get coordinates of a city
def get_city_coordinates(city_name):
    url = "https://maps.gomaps.pro/maps/api/geocode/json"
    params = {
        "address": city_name,
        "key": API_KEY
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        results = response.json().get("results", [])
        if results:
            location = results[0]["geometry"]["location"]
            return location["lat"], location["lng"]
        else:
            print("City not found. Using default coordinates.")
            return None, None
    else:
        print("Error fetching city coordinates:", response.status_code, response.text)
        return None, None

# Get city name input from user
city_name = input("Enter the city name (e.g., Dehradun): ")

# Get coordinates for the given city
latitude, longitude = get_city_coordinates(city_name)

# If no valid coordinates, exit
if latitude is None or longitude is None:
    print("Could not get coordinates for the given city. Exiting.")
    exit()

search_radius = 5000  # Radius for initial search in meters
nearby_radius = 500  # Radius to check for nearby bakeries around butcher shops in meters

# Alternative types/keywords for butcher shops
butcher_keywords = ["meat_shop", "chicken_shop", "mutton_shop"]

# Function to search for places using GoMaps Places API
def search_places_nearby(keyword, lat, lng, radius):
    url = "https://maps.gomaps.pro/maps/api/place/nearbysearch/json"
    params = {
        "location": f"{lat},{lng}",
        "radius": radius,
        "keyword": keyword,
        "key": API_KEY
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json().get("results", [])
    else:
        print("Error:", response.status_code, response.text)
        return []

# Haversine formula to calculate the distance between two coordinates in meters
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Radius of Earth in meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c  # Return distance in meters

# Combined list to store butcher shop results from all keywords
butcher_shops = []

# Perform search for each butcher-related keyword
for keyword in butcher_keywords:
    butcher_shops.extend(search_places_nearby(keyword, latitude, longitude, search_radius))

# Remove duplicates based on place_id
unique_butcher_shops = {shop["place_id"]: shop for shop in butcher_shops}.values()

# Prepare data for machine learning model
data = []
labels = []

# For each butcher shop, search for nearby bakeries
for butcher in unique_butcher_shops:
    butcher_name = butcher.get("name")
    butcher_location = butcher.get("vicinity")
    butcher_lat = butcher["geometry"]["location"]["lat"]
    butcher_lng = butcher["geometry"]["location"]["lng"]

    # Search for bakeries near each butcher shop
    bakeries_near_butcher = search_places_nearby("bakery", butcher_lat, butcher_lng, nearby_radius)

    for bakery in bakeries_near_butcher:
        bakery_lat = bakery["geometry"]["location"]["lat"]
        bakery_lng = bakery["geometry"]["location"]["lng"]

        # Calculate distance between bakery and butcher shop
        distance = haversine(bakery_lat, bakery_lng, butcher_lat, butcher_lng)

        # Label 1 if bakery is near a butcher shop, 0 otherwise
        label = 1 if distance <= nearby_radius else 0  # If within 500 meters, label as 1 (high risk)

        # Append features (distance to butcher shop) and labels
        data.append([distance])
        labels.append(label)

# Train the machine learning model
X = np.array(data)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest model for classification
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model accuracy
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy: {accuracy:.2f}")

# Predicting the risk for a new bakery based on its distance from a butcher shop
def predict_risk(distance):
    risk = model.predict([[distance]])
    if risk == 1:
        return "High Risk (Near Butcher Shop) - Offer higher premium."
    else:
        return "Low Risk (Not Near Butcher Shop) - Offer normal premium."

# Process the butcher and bakery data
for butcher in unique_butcher_shops:
    butcher_name = butcher.get("name")
    butcher_location = butcher.get("vicinity")
    butcher_lat = butcher["geometry"]["location"]["lat"]
    butcher_lng = butcher["geometry"]["location"]["lng"]

    print(f"\nButcher Shop: {butcher_name}, Location: {butcher_location}")

    # Search for bakeries near each butcher shop
    bakeries_near_butcher = search_places_nearby("bakery", butcher_lat, butcher_lng, nearby_radius)

    if bakeries_near_butcher:
        print("Nearby Bakeries:")
        for bakery in bakeries_near_butcher:
            bakery_name = bakery.get("name")
            bakery_location = bakery.get("vicinity")
            bakery_lat = bakery["geometry"]["location"]["lat"]
            bakery_lng = bakery["geometry"]["location"]["lng"]

            # Calculate distance between bakery and butcher shop
            distance = haversine(bakery_lat, bakery_lng, butcher_lat, butcher_lng)

            # Use ML model to predict the risk level
            model_outcome = predict_risk(distance)

            print(f" - Bakery Name: {bakery_name}, Address: {bakery_location}, Distance: {distance:.2f} meters, Model Outcome: {model_outcome}")
    else:
        print("No bakeries found near this butcher shop.")


In [ ]:
#COMPLETE CODE FINAL dehradun large dataset
import requests
import math
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import numpy as np

# Google Maps Places API Key
API_KEY = "AlzaSyOkVMhcZDvHkm2ykwxh51Z-EM_wbk1mAWx"  # Replace with your actual API key


# Define the coordinates of Rajpur Road, Dehradun
latitude, longitude = 30.3450, 78.0485
search_radius = 5000  # Radius for initial search in meters
nearby_radius = 500  # Radius to check for nearby bakeries around butcher shops in meters

# Alternative types/keywords for butcher shops
butcher_keywords = ["meat_shop", "chicken_shop", "mutton_shop"]

# Function to search for places using Google Maps Places API
def search_places_nearby(keyword, lat, lng, radius):
    url = "https://maps.gomaps.pro/maps/api/place/nearbysearch/json"
    params = {
        "location": f"{lat},{lng}",
        "radius": radius,
        "keyword": keyword,
        "key": API_KEY
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json().get("results", [])
    else:
        print("Error:", response.status_code, response.text)
        return []

# Haversine formula to calculate the distance between two coordinates in meters
def haversine(lat1, lon1, lat2, lon2):
    # Radius of Earth in meters
    R = 6371000
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    
    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return R * c  # Return distance in meters

# Combined list to store butcher shop results from all keywords
butcher_shops = []

# Perform search for each butcher-related keyword
for keyword in butcher_keywords:
    butcher_shops.extend(search_places_nearby(keyword, latitude, longitude, search_radius))

# Remove duplicates based on place_id
unique_butcher_shops = {shop["place_id"]: shop for shop in butcher_shops}.values()

# Prepare data for machine learning model
data = []
labels = []

# For each butcher shop, search for nearby bakeries
for butcher in unique_butcher_shops:
    butcher_name = butcher.get("name")
    butcher_location = butcher.get("vicinity")
    butcher_lat = butcher["geometry"]["location"]["lat"]
    butcher_lng = butcher["geometry"]["location"]["lng"]
    
    # Search for bakeries near each butcher shop
    bakeries_near_butcher = search_places_nearby("bakery", butcher_lat, butcher_lng, nearby_radius)
    
    for bakery in bakeries_near_butcher:
        bakery_lat = bakery["geometry"]["location"]["lat"]
        bakery_lng = bakery["geometry"]["location"]["lng"]
        
        # Calculate distance between bakery and butcher shop
        distance = haversine(bakery_lat, bakery_lng, butcher_lat, butcher_lng)
        
        # Label 1 if bakery is near a butcher shop, 0 otherwise
        label = 1 if distance <= nearby_radius else 0  # If within 100 meters, label as 1 (high risk)
        
        # Append features (distance to butcher shop) and labels
        data.append([distance])
        labels.append(label)

# Train the machine learning model
X = np.array(data)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest model for classification
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model accuracy
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy: {accuracy:.2f}")

# Predicting the risk for a new bakery based on its distance from a butcher shop
def predict_risk(distance):
    risk = model.predict([[distance]])
    if risk == 1:
        return "High Risk (Near Butcher Shop) - Offer higher premium."
    else:
        return "Low Risk (Not Near Butcher Shop) - Offer normal premium."

# Process the butcher and bakery data
for butcher in unique_butcher_shops:
    butcher_name = butcher.get("name")
    butcher_location = butcher.get("vicinity")
    butcher_lat = butcher["geometry"]["location"]["lat"]
    butcher_lng = butcher["geometry"]["location"]["lng"]
    
    print(f"\nButcher Shop: {butcher_name}, Location: {butcher_location}")
    
    # Search for bakeries near each butcher shop
    bakeries_near_butcher = search_places_nearby("bakery", butcher_lat, butcher_lng, nearby_radius)
    
    if bakeries_near_butcher:
        print("Nearby Bakeries:")
        for bakery in bakeries_near_butcher:
            bakery_name = bakery.get("name")
            bakery_location = bakery.get("vicinity")
            bakery_lat = bakery["geometry"]["location"]["lat"]
            bakery_lng = bakery["geometry"]["location"]["lng"]
            
            # Calculate distance between bakery and butcher shop
            distance = haversine(bakery_lat, bakery_lng, butcher_lat, butcher_lng)
            
            # Use ML model to predict the risk level
            model_outcome = predict_risk(distance)
            
            print(f" - Bakery Name: {bakery_name}, Address: {bakery_location}, Distance: {distance:.2f} meters, Model Outcome: {model_outcome}")
    else:
        print("No bakeries found near this butcher shop.")
